In [1]:
from neo4j import GraphDatabase
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM, OllamaLLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation.prompts import ERExtractionTemplate
from dotenv import load_dotenv
import pandas as pd
from langchain_core.documents import Document
from langchain_community.llms import Ollama
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.schema import get_schema
from neo4j_graphrag.generation import GraphRAG
import json
import numpy as np
import ast

import os, time, asyncio, glob, csv

d:\GitHub\muict_thai_legal_RAG_GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(dotenv_path="../.env")

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE_NAME = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [22]:
df = pd.read_parquet('../data/processed/ref_vectors_sparse_nitibench-ccl-bge-m3.parquet')
df.head()

,id,law_name,section_num,text,dense_vector,sparse_vector,reference_laws
0,0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,132,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,"[-0.03570556640625, 0.004070281982421001, -0.0...","{""221767"": 0.060760498046875, ""70390"": 0.15673...",[{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน...
1,1,ประมวลกฎหมายแพ่งและพาณิชย์,1598/5,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,"[0.0345458984375, 0.049072265625, -0.020095825...","{""130047"": 0.0031299591064453125, ""57467"": 0.0...",[]
2,2,ประมวลกฎหมายแพ่งและพาณิชย์,876,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,"[-0.039886474609375, 0.05941772460937501, 0.00...","{""167120"": 0.015411376953125, ""382"": 0.0497741...",[]
3,3,ประมวลกฎหมายแพ่งและพาณิชย์,1030,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,"[-0.029144287109375003, 0.024978637695312, -0....","{""130047"": 0.11090087890625, ""57467"": 0.025955...",[]
4,4,ประมวลกฎหมายแพ่งและพาณิชย์,849,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,"[0.011123657226562, 0.0675048828125, -0.041503...","{""130047"": 0.00732421875, ""57467"": 0.020690917...",[]


In [25]:
# 1. ตั้งค่าการเชื่อมต่อ Neo4j
load_dotenv(dotenv_path="../.env")

class Neo4jLawImporter:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def process_embedding(self, val):
        if isinstance(val, (list, np.ndarray)):
            return [float(x) for x in val]
        if isinstance(val, str):
            clean_str = val.replace('[', '').replace(']', '').replace('\n', ' ').strip()
            return [float(x) for x in re.split(r'\s+', clean_str) if x]
        return []

    def process_reference_laws(self, val):
        """แก้ไขจุดที่เกิด Error: เช็กว่าเป็นค่าว่างในรูปแบบต่างๆ หรือไม่"""
        # 1. เช็กว่าเป็น None หรือ NaN (ใช้ pd.isna ที่รองรับทั้งคู่)
        if pd.isna(val) if isinstance(val, (float, int, type(None))) else False:
            return []
        
        # 2. ถ้าเป็น list หรือ numpy array
        if isinstance(val, (list, np.ndarray)):
            # ถ้าขนาดเป็น 0 ให้คืนค่า list ว่าง
            return list(val) if len(val) > 0 else []
        
        # 3. ถ้าเป็น string (เช่น "[]" หรือ "[{...}]")
        if isinstance(val, str):
            val = val.strip()
            if val == "" or val == "[]":
                return []
            try:
                res = ast.literal_eval(val)
                return res if isinstance(res, list) else []
            except:
                return []
        
        return []

    def import_data(self, df):
        self.driver.execute_query(
            "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Section) REQUIRE s.law_id IS UNIQUE"
        )

        for _, row in df.iterrows():
            law_id = f"{row['law_name']}_{row['section_num']}"
            embedding = self.process_embedding(row['dense_vector'])
            
            # Insert Node หลัก
            self.driver.execute_query(
                """
                MERGE (s:Section {law_id: $law_id})
                SET s.id = $id,
                    s.law_name = $law_name,
                    s.section_num = $section_num,
                    s.text = $text,
                    s.embedding = $embedding
                """,
                law_id=law_id,
                id=row['id'],
                law_name=row['law_name'],
                section_num=row['section_num'],
                text=row['text'],
                embedding=embedding,
                database_="neo4j",
            )

            # จัดการความสัมพันธ์
            ref_laws = self.process_reference_laws(row['reference_laws'])
            for ref in ref_laws:
                # ตรวจสอบว่า ref เป็น dict และมี key ครบถ้วน
                if isinstance(ref, dict) and 'law_name' in ref and 'section_num' in ref:
                    target_id = f"{ref['law_name']}_{ref['section_num']}"
                    
                    self.driver.execute_query(
                        """
                        MATCH (source:Section {law_id: $source_id})
                        MERGE (target:Section {law_id: $target_id})
                        ON CREATE SET target.law_name = $t_name, 
                                      target.section_num = $t_sec
                        MERGE (source)-[:REFERENCES_TO]->(target)
                        """,
                        source_id=law_id,
                        target_id=target_id,
                        t_name=ref['law_name'],
                        t_sec=ref['section_num'],
                        database_="neo4j",
                    )

# การรันโค้ด
importer = Neo4jLawImporter(URI, USERNAME, PASSWORD)
importer.import_data(df)